In [15]:
import os
import json
import joblib
import numpy as np
import pandas as pd
from pathlib import Path
from groq import Groq
from dotenv import load_dotenv

# Resolve project root robustly from current working directory
cwd = Path.cwd().resolve()
PROJECT_ROOT = next((d for d in [cwd, *cwd.parents] if (d / "models").exists() and (d / "data").exists()), cwd)
MODELS_DIR = PROJECT_ROOT / "models"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load API key from .env file
load_dotenv(PROJECT_ROOT / ".env")

groq_api_key = os.environ.get("GROQ_API_KEY")
if not groq_api_key:
    raise ValueError(
        "GROQ_API_KEY is not set. Add it to .env as GROQ_API_KEY=your_key and rerun this cell."
    )

client = Groq(api_key=groq_api_key)

# Preferred model (override in .env)
GROQ_MODEL = os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant")
# Fallbacks if a model is decommissioned
GROQ_MODEL_FALLBACKS = [
    GROQ_MODEL,
    "llama-3.1-8b-instant",
    "llama-3.3-70b-versatile",
    "mixtral-8x7b-32768",
]
# Preserve order while removing duplicates
GROQ_MODEL_FALLBACKS = list(dict.fromkeys(GROQ_MODEL_FALLBACKS))

# Load everything we need
model = joblib.load(MODELS_DIR / 'xgb_model.pkl')
explainer = joblib.load(MODELS_DIR / 'shap_explainer.pkl')
feature_cols = joblib.load(MODELS_DIR / 'feature_cols.pkl')
shap_values = joblib.load(MODELS_DIR / 'shap_values_test.pkl')

test = pd.read_csv(PROCESSED_DIR / 'test_features.csv')
test_last = test.groupby('engine_id').last().reset_index()
X_test_last = test_last[feature_cols]

print("Project root:", PROJECT_ROOT)
print("Everything loaded.")
print(f"Test engines available: {len(test_last)}")




Project root: /Users/heshangamage/predictive-maintenance-industrial-ai
Everything loaded.
Test engines available: 100


In [16]:
def classify_risk(predicted_rul):
    """
    Convert predicted RUL into a risk classification.
    Thresholds based on the 30-40 cycle warning window
    we identified in data exploration.
    """
    if predicted_rul <= 15:
        return 'CRITICAL'
    elif predicted_rul <= 30:
        return 'HIGH'
    elif predicted_rul <= 60:
        return 'MEDIUM'
    else:
        return 'LOW'

# Test it
for rul in [5, 20, 45, 90]:
    print(f"RUL {rul:3d} cycles → {classify_risk(rul)}")

RUL   5 cycles → CRITICAL
RUL  20 cycles → HIGH
RUL  45 cycles → MEDIUM
RUL  90 cycles → LOW


In [17]:
def get_top_contributions(engine_idx, shap_vals, feature_names, top_n=5):
    engine_shap = shap_vals[engine_idx]
    shap_series = pd.Series(engine_shap, index=feature_names)
    top_features = shap_series.abs().sort_values(ascending=False).head(top_n)
    
    contributions = []
    for feature in top_features.index:
        shap_val = shap_series[feature]
        contributions.append({
            'feature': feature,
            'shap_value': round(float(shap_val), 4),
            'direction': 'increasing risk' if shap_val < 0 else 'decreasing risk',
            'impact': 'HIGH' if abs(shap_val) > 5 else 'MEDIUM' if abs(shap_val) > 2 else 'LOW'
        })
    
    return contributions


def format_shap_for_prompt(contributions):
    """Convert SHAP contributions to readable text for the LLM."""
    lines = []
    sensor_names = {
        's2': 'LPC outlet temperature',
        's3': 'HPC outlet temperature', 
        's4': 'LPT outlet temperature',
        's7': 'HPC outlet pressure',
        's8': 'Physical fan speed',
        's9': 'Core speed',
        's11': 'HPC outlet static pressure',
        's12': 'Fuel flow to pressure ratio',
        's13': 'Corrected fan speed',
        's14': 'Corrected core speed',
        's15': 'Bypass ratio',
        's17': 'Bleed enthalpy',
        's20': 'HPT coolant bleed',
        's21': 'LPT coolant bleed'
    }
    
    for c in contributions:
        # Parse feature name e.g. s4_mean_5
        parts = c['feature'].split('_')
        sensor_code = parts[0]
        sensor_desc = sensor_names.get(sensor_code, sensor_code)
        
        if len(parts) == 3:
            stat, window = parts[1], parts[2]
            readable = f"{sensor_desc} ({window}-cycle rolling {stat})"
        else:
            readable = sensor_desc
            
        direction = "ABNORMAL - pushing toward failure" if c['direction'] == 'increasing risk' else "normal"
        lines.append(
            f"  - {readable}: {direction} "
            f"(magnitude: {abs(c['shap_value']):.3f}, impact: {c['impact']})"
        )
    
    return "\n".join(lines)


# Test both functions
contributions = get_top_contributions(0, shap_values, feature_cols)
print(format_shap_for_prompt(contributions))

  - LPT outlet temperature (5-cycle rolling mean): normal (magnitude: 4.535, impact: MEDIUM)
  - HPC outlet static pressure (5-cycle rolling mean): normal (magnitude: 3.712, impact: MEDIUM)
  - Corrected core speed (30-cycle rolling mean): normal (magnitude: 3.604, impact: MEDIUM)
  - LPC outlet temperature (10-cycle rolling mean): normal (magnitude: 3.193, impact: MEDIUM)
  - HPC outlet temperature (30-cycle rolling mean): ABNORMAL - pushing toward failure (magnitude: 2.906, impact: MEDIUM)


In [18]:
WORK_ORDER_PROMPT = """You are an industrial maintenance planning assistant for a fleet of aircraft turbofan engines. A predictive AI system has flagged an engine for maintenance based on sensor data analysis.

PREDICTION SUMMARY:
- Asset ID: {asset_id}
- Current Operating Cycle: {current_cycle}
- Predicted Remaining Useful Life: {predicted_rul} cycles
- Risk Classification: {risk_level}

SENSOR ANALYSIS (from SHAP explainability):
{shap_contributions}

Based on this analysis, generate a maintenance work order as a JSON object with EXACTLY these fields:

{{
  "asset_id": "{asset_id}",
  "priority": "<low | medium | high | critical>",
  "predicted_failure_window": "<human readable, e.g. within 15 cycles>",
  "affected_components": ["<list of physical engine components to inspect>"],
  "recommended_parts": ["<list of likely replacement parts>"],
  "estimated_hours": <number between 2 and 40>,
  "required_technician_skill": "<level 1 | level 2 | level 3 | specialist>",
  "safety_precautions": ["<list of safety steps before work begins>"],
  "summary_for_technician": "<2-3 sentences in plain English the technician reads on their mobile device>"
}}

RULES:
- Map sensor anomalies to physical components using your knowledge of turbofan engine architecture
- s4 anomalies relate to Low Pressure Turbine and hot section components
- s11 and s7 anomalies relate to High Pressure Compressor
- s12 anomalies relate to fuel system and combustor
- s15 anomalies relate to fan and bypass duct
- Priority must match risk level exactly: CRITICAL->critical, HIGH->high, MEDIUM->medium, LOW->low
- Estimated hours: 2-4 for inspection only, 6-12 for component replacement, 16-40 for major overhaul
- CRITICAL and HIGH risk engines need specialist or level 3 technicians
- Respond with ONLY the JSON object. No markdown, no explanation, no preamble."""

In [19]:
def generate_work_order(engine_id, engine_idx):
    """
    For a given engine, generate a full maintenance work order
    using the ML prediction and SHAP explanation as context.
    """
    
    # Get prediction
    engine_features = X_test_last.iloc[[engine_idx]]
    predicted_rul = float(np.clip(model.predict(engine_features)[0], 0, 125))
    risk_level = classify_risk(predicted_rul)
    current_cycle = int(test_last.iloc[engine_idx]['cycle'])
    
    # Get SHAP contributions
    contributions = get_top_contributions(engine_idx, shap_values, feature_cols)
    formatted_shap = format_shap_for_prompt(contributions)
    
    # Build prompt
    prompt = WORK_ORDER_PROMPT.format(
        asset_id=f"ENGINE-{int(engine_id):03d}",
        current_cycle=current_cycle,
        predicted_rul=round(predicted_rul, 1),
        risk_level=risk_level,
        shap_contributions=formatted_shap
    )
    
    # Call Groq with model fallback for decommissioned models
    model_fallbacks = globals().get(
        "GROQ_MODEL_FALLBACKS",
        [
            os.environ.get("GROQ_MODEL", "llama-3.1-8b-instant"),
            "llama-3.1-8b-instant",
            "llama-3.3-70b-versatile",
            "mixtral-8x7b-32768",
        ],
    )
    model_fallbacks = list(dict.fromkeys(model_fallbacks))

    response = None
    last_error = None
    for model_name in model_fallbacks:
        try:
            response = client.chat.completions.create(
                model=model_name,
                max_tokens=1000,
                temperature=0.2,
                messages=[{"role": "user", "content": prompt}],
            )
            break
        except Exception as e:
            last_error = e
            if "model_decommissioned" in str(e) or "decommissioned" in str(e).lower():
                continue
            raise

    if response is None:
        raise RuntimeError(
            f"All candidate Groq models failed: {model_fallbacks}. Last error: {last_error}"
        )

    raw = (response.choices[0].message.content or "").strip()

    
    # Defensive JSON parsing
    if raw.startswith("```"):
        raw = raw.split("```")[1]
        if raw.startswith("json"):
            raw = raw[4:]
        raw = raw.strip()

    # Extra safety - extract just the JSON object
    start = raw.find('{')
    end = raw.rfind('}') + 1
    if start != -1 and end != 0:
        raw = raw[start:end]
    
    work_order = json.loads(raw)
    
    # Add metadata
    work_order['predicted_rul'] = round(predicted_rul, 1)
    work_order['risk_level'] = risk_level
    work_order['current_cycle'] = current_cycle
    
    return work_order


# Test on engine index 0
print("Generating work order for engine 1...")
wo = generate_work_order(
    engine_id=test_last.iloc[0]['engine_id'],
    engine_idx=0
)

print(json.dumps(wo, indent=2))




Generating work order for engine 1...
{
  "asset_id": "ENGINE-001",
  "priority": "low",
  "predicted_failure_window": "within 117 cycles",
  "affected_components": [
    "HPC (High Pressure Compressor) vanes and blades"
  ],
  "recommended_parts": [
    "HPC vane and blade set"
  ],
  "estimated_hours": 6,
  "required_technician_skill": "level 3",
  "safety_precautions": [
    "Ensure proper engine shutdown and cooling before inspection",
    "Wear personal protective equipment"
  ],
  "summary_for_technician": "Inspect and replace HPC vanes and blades due to abnormal HPC outlet temperature. Perform inspection and replacement within the next 6 hours.",
  "predicted_rul": 117.1,
  "risk_level": "LOW",
  "current_cycle": 31
}


In [20]:
# Find highest risk engine
predictions = np.clip(model.predict(X_test_last), 0, 125)
highest_risk_idx = np.argmin(predictions)
highest_risk_engine_id = test_last.iloc[highest_risk_idx]['engine_id']

print(f"Highest risk engine: ENGINE-{int(highest_risk_engine_id):03d}")
print(f"Predicted RUL: {predictions[highest_risk_idx]:.1f} cycles")
print(f"Risk level: {classify_risk(predictions[highest_risk_idx])}")
print()

wo_critical = generate_work_order(highest_risk_engine_id, highest_risk_idx)
print(json.dumps(wo_critical, indent=2))

Highest risk engine: ENGINE-034
Predicted RUL: 6.2 cycles
Risk level: CRITICAL

{
  "asset_id": "ENGINE-034",
  "priority": "critical",
  "predicted_failure_window": "within 6.2 cycles",
  "affected_components": [
    "Low Pressure Turbine (LPT) hot section",
    "High Pressure Compressor (HPC) blades",
    "Combustor",
    "Bypass duct"
  ],
  "recommended_parts": [
    "LPT hot section components",
    "HPC blades",
    "Combustor liner",
    "Bypass duct seals"
  ],
  "estimated_hours": 24,
  "required_technician_skill": "specialist",
  "safety_precautions": [
    "Ensure proper engine shutdown procedures are followed",
    "Use personal protective equipment (PPE) when working with hot components",
    "Verify proper ventilation in the work area"
  ],
  "summary_for_technician": "Engine 034 is predicted to fail within 6.2 cycles due to multiple sensor anomalies. Perform a thorough inspection and replacement of affected components to prevent engine failure.",
  "predicted_rul": 6.2,


In [21]:
def get_fleet_status():
    """
    Generate predictions and risk levels for all test engines.
    Only generate full work orders for HIGH and CRITICAL engines.
    """
    fleet = []
    
    for idx, row in test_last.iterrows():
        engine_id = row['engine_id']
        predicted_rul = float(np.clip(
            model.predict(X_test_last.iloc[[idx]])[0], 0, 125
        ))
        risk_level = classify_risk(predicted_rul)
        
        fleet.append({
            'engine_id': f"ENGINE-{int(engine_id):03d}",
            'current_cycle': int(row['cycle']),
            'predicted_rul': round(predicted_rul, 1),
            'risk_level': risk_level
        })
    
    fleet_df = pd.DataFrame(fleet)
    fleet_df = fleet_df.sort_values('predicted_rul')
    
    return fleet_df


fleet = get_fleet_status()

print("Fleet Status Summary:")
print(fleet['risk_level'].value_counts())
print()
print("Top 10 highest risk engines:")
print(fleet.head(10).to_string(index=False))

Fleet Status Summary:
risk_level
LOW         64
MEDIUM      14
HIGH        13
CRITICAL     9
Name: count, dtype: int64

Top 10 highest risk engines:
 engine_id  current_cycle  predicted_rul risk_level
ENGINE-034            203            6.2   CRITICAL
ENGINE-076            205            6.4   CRITICAL
ENGINE-081            213            7.3   CRITICAL
ENGINE-082            162            8.1   CRITICAL
ENGINE-068            187            8.1   CRITICAL
ENGINE-020            184            9.1   CRITICAL
ENGINE-031            196           11.0   CRITICAL
ENGINE-035            198           12.6   CRITICAL
ENGINE-042            156           12.6   CRITICAL
ENGINE-049            303           15.1       HIGH


In [22]:
import time

high_risk = fleet[fleet['risk_level'].isin(['CRITICAL', 'HIGH'])]
print(f"Generating work orders for {len(high_risk)} high/critical engines...")

work_orders = {}

for _, row in high_risk.iterrows():
    engine_id_str = row['engine_id']
    engine_num = int(engine_id_str.split('-')[1])
    
    # Find index in test_last
    idx = test_last[test_last['engine_id'] == engine_num].index[0]
    idx_in_array = test_last.index.get_loc(idx)
    
    print(f"  Generating for {engine_id_str}...", end=' ')
    
    try:
        wo = generate_work_order(engine_num, idx_in_array)
        work_orders[engine_id_str] = wo
        print(f"✓ ({wo['priority']} priority)")
        time.sleep(0.5)  # be gentle with the API
    except Exception as e:
        print(f"✗ Error: {e}")

# Save to disk
os.makedirs('../data/processed', exist_ok=True)
with open('../data/processed/work_orders_cache.json', 'w') as f:
    json.dump(work_orders, f, indent=2)

fleet.to_csv('../data/processed/fleet_status.csv', index=False)

print(f"\nSaved {len(work_orders)} work orders to cache.")
print("Saved fleet status to fleet_status.csv")

Generating work orders for 22 high/critical engines...
  Generating for ENGINE-034... ✓ (critical priority)
  Generating for ENGINE-076... ✓ (critical priority)
  Generating for ENGINE-081... ✓ (critical priority)
  Generating for ENGINE-082... ✓ (critical priority)
  Generating for ENGINE-068... ✓ (critical priority)
  Generating for ENGINE-020... ✓ (critical priority)
  Generating for ENGINE-031... ✓ (critical priority)
  Generating for ENGINE-035... ✓ (critical priority)
  Generating for ENGINE-042... ✓ (critical priority)
  Generating for ENGINE-049... ✓ (high priority)
  Generating for ENGINE-056... ✓ (high priority)
  Generating for ENGINE-066... ✓ (high priority)
  Generating for ENGINE-036... ✓ (high priority)
  Generating for ENGINE-024... ✓ (high priority)
  Generating for ENGINE-092... ✓ (high priority)
  Generating for ENGINE-100... ✓ (high priority)
  Generating for ENGINE-041... ✓ (high priority)
  Generating for ENGINE-061... ✓ (high priority)
  Generating for ENGINE-090

In [23]:
# Load and inspect
with open('../data/processed/work_orders_cache.json', 'r') as f:
    cached = json.load(f)

print(f"Cached work orders: {len(cached)}")
print()

# Show summary of what was generated
for engine_id, wo in list(cached.items())[:3]:
    print(f"{engine_id}:")
    print(f"  Priority: {wo['priority']}")
    print(f"  Predicted RUL: {wo['predicted_rul']} cycles")
    print(f"  Estimated hours: {wo['estimated_hours']}")
    print(f"  Technician: {wo['required_technician_skill']}")
    print(f"  Summary: {wo['summary_for_technician'][:80]}...")
    print()

Cached work orders: 22

ENGINE-034:
  Priority: critical
  Predicted RUL: 6.2 cycles
  Estimated hours: 24
  Technician: level 3
  Summary: Engine-034 is predicted to fail within 6.2 cycles due to multiple sensor anomali...

ENGINE-076:
  Priority: critical
  Predicted RUL: 6.4 cycles
  Estimated hours: 24
  Technician: specialist
  Summary: Engine 076 is predicted to fail within 6.4 cycles. Perform a thorough inspection...

ENGINE-081:
  Priority: critical
  Predicted RUL: 7.3 cycles
  Estimated hours: 24
  Technician: specialist
  Summary: Perform a thorough inspection and replacement of affected components in ENGINE-0...

